In [23]:
from tavily import TavilyClient
from langchain.tools import tool 

In [24]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq


In [25]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Model Initialization
llm = ChatGroq(
    model="llama-3.1-8b-instant",        
    temperature=0,
    api_key=GROQ_API_KEY,
    max_tokens=1024,        
    
)

In [26]:
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [27]:
@tool
def web_search(query : str) -> str:
    """Search the web for recent and reliable information on a topic . Returns Titles , URLs and snippets."""
    results = tavily.search(query=query,max_results=5)

    out = []

    for r in results['results']:
        out.append(
            f"Title: {r['title']}\nURL: {r['url']}\nSnippet: {r['content'][:300]}\n"
        )
    
    return "\n----\n".join(out)

In [28]:
from langchain.agents import create_agent

In [29]:
def build_search_agent():
    return create_agent(
        model= llm,
        tools=[web_search],
       
    )

In [30]:

state = {}

#search agent working 
print("\n"+" ="*50)
print("step 1 - search agent is working ...")
print("="*50)
search_agent = build_search_agent()
search_result = search_agent.invoke({
    "messages" : [("user", f"Find recent, reliable and detailed information about: {'Future of LLM in Tech Industry'}")]
})
# state["search_results"] = search_result

state["search_results"] = "\n\n".join(
    m.content for m in search_result["messages"]
    if m.__class__.__name__ in ("ToolMessage", "AIMessage") and m.content
)



print("\n search result ",state['search_results'])

= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =

step 1 - search agent is working ...

==================================================

search result  Title: Future of Large Language Models - GeeksforGeeks
URL: https://www.geeksforgeeks.org/machine-learning/future-of-large-language-models
Snippet: # Future of Large Language Models. In the last few years, the development of artificial intelligence has 
been in significant demand, with the emergence of ****Large Language Models (LLMs)****. This streamlined model 
entails advanced machine learning methods, has transformed natural language procedur

----
Title: Top LLM Trends 2025: What's the Future of LLMs - Turing
URL: https://www.turing.com/resources/top-llm-trends
Snippet: LLMs are evolving from basic assistants to (https://www.turing.com/resources/ai-agentic-workflows) that 
can act independently. [Ethical AI](https://www.turing.com/resources/implementing-security-guardrails-for-llms) 
became a shared responsibility across developers, researchers, an

----
Title: What comes after the LLMs? The AI industry is organized around ...
URL: 
https://www.facebook.com/technologyreview/posts/what-comes-after-the-llmsthe-ai-industry-is-organized-around-large-
language-mode/1283847436937737
Snippet: ## MIT Technology Review's Post. ### **MIT Technology Review**. What comes after the LLMs? The AI industry
is organized around large language models, from tools to business models. But many researchers believe the next 
breakthroughs may not look like language models at all. Join us live tomorrow, Ma

----
Title: Real-world Applications of Large Language Models | LLM Tutorial
URL: https://www.youtube.com/watch?v=zQ-pLyp9Kp0
Snippet: Real-world Applications of Large Language Models | LLM Tutorial | Future of Data and AI | Conference
Data Science Dojo
122000 subscribers
66 likes
4744 views
16 Mar 2023
Custom Large Language Models can improve your business in a number of ways. If you're not already using LLMs in 
your business, I e

----
Title: AI LLM Evolution and the future of IT jobs! | by Seetaram N T - Medium
URL: https://medium.com/@seetaram.nt/ai-llm-evolution-and-the-future-of-it-jobs-21440b36e25a
Snippet: AI LLM Evolution and the future of IT jobs ... This can be used by people as reference to plan their 
career in IT Industry going forward!


Based on the search results, it appears that the future of Large Language Models (LLMs) in the tech industry is 
expected to be shaped by several trends and developments. Some of the key points that can be inferred from the 
search results are:

1. **Evolution of LLMs**: LLMs are expected to evolve from basic assistants to autonomous agents that can act 
independently.
2. **Ethical AI**: The development of LLMs raises ethical concerns, and there is a growing need for developers, 
researchers, and organizations to prioritize ethical AI practices.
3. **Real-world applications**: LLMs have the potential to improve various aspects of business and industry, 
including customer service, content creation, and decision-making.
4. **Career implications**: The evolution of LLMs may impact the job market, with some roles becoming redundant and
new ones emerging.
5. **Next breakthroughs**: Researchers believe that the next breakthroughs in AI may not come from language models 
but from other areas, such as computer vision or reinforcement learning.

Overall, the future of LLMs in the tech industry is expected to be shaped by a combination of technological 
advancements, ethical considerations, and real-world applications.

In [31]:
search_result

{'messages': [HumanMessage(content='Find recent, reliable and detailed information about: Future of LLM in Tech Industry', additional_kwargs={}, response_metadata={}, id='e2c4ddfa-37d3-4f62-8431-f32ca06dac60'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3mw6wcgt9', 'function': {'arguments': '{"query":"Future of LLM in Tech Industry"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 252, 'total_tokens': 273, 'completion_time': 0.036833951, 'completion_tokens_details': None, 'prompt_time': 0.030370322, 'prompt_tokens_details': None, 'queue_time': 0.005912943, 'total_time': 0.067204273}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_03e8423237', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef809-9b2d-7dc2-9945-c088e8c8e0be-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'Future of LLM in Tec

In [32]:
import requests 
import requests
from rich import print
from bs4 import BeautifulSoup
from readability import Document
import trafilatura
import re 

In [33]:
@tool
def scrape_url(url: str) -> str:
    """
    Scrape and extract clean readable content from a URL.
    Uses multiple extraction strategies for better reliability.
    """

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.google.com/",
    }

    try:
        # ── Fetch page ─────────────────────────────────────
        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

        response.raise_for_status()

        html = response.text

        # ──────────────────────────────────────────────────
        # Strategy 1 → trafilatura (BEST for articles/blogs)
        # ──────────────────────────────────────────────────
        extracted = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False
        )

        if extracted and len(extracted.strip()) > 200:
            cleaned = re.sub(r'\s+', ' ', extracted)
            return cleaned[:5000]

        # ──────────────────────────────────────────────────
        # Strategy 2 → readability
        # ──────────────────────────────────────────────────
        doc = Document(html)
        clean_html = doc.summary()

        soup = BeautifulSoup(clean_html, "html.parser")

        for tag in soup([
            "script",
            "style",
            "nav",
            "footer",
            "header",
            "aside",
            "form"
        ]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)

        if text and len(text.strip()) > 200:
            cleaned = re.sub(r'\s+', ' ', text)
            return cleaned[:5000]

        # ──────────────────────────────────────────────────
        # Strategy 3 → fallback full page extraction
        # ──────────────────────────────────────────────────
        soup = BeautifulSoup(html, "html.parser")

        for tag in soup([
            "script",
            "style",
            "nav",
            "footer",
            "header",
            "aside",
            "form"
        ]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)

        cleaned = re.sub(r'\s+', ' ', text)

        if cleaned:
            return cleaned[:5000]

        return "Could not extract meaningful content from the page."

    except requests.exceptions.Timeout:
        return "Request timed out while scraping the URL."

    except requests.exceptions.HTTPError as e:
        return f"HTTP error occurred: {str(e)}"

    except Exception as e:
        return f"Could not scrape URL: {str(e)}"

In [34]:
def build_reader_agent():
    return create_agent(
        model= llm,
        tools=[scrape_url],

    )

In [35]:
#step 2 - reader agent 
print("\n"+" ="*50)
print("step 2 - Reader agent is scraping top resources ...")
print("="*50)

reader_agent = build_reader_agent()
search_results_str = str(state.get('search_results', ''))

reader_result = reader_agent.invoke({
    "messages": [
        ("user", 
         f"Based on the following search results about '{'Future of LLM in Tech Industry'}', "
         f"pick the most relevant URL(s) and scrape them for deeper content.\n\n"
         f"Search Results:\n{search_results_str}"
        )
    ]
})

state['scraped_content'] = reader_result['messages']

print("\nscraped content: \n", state['scraped_content'])



= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =

step 2 - Reader agent is scraping top resources ...

==================================================

scraped content: 

[
    HumanMessage(
        content="Based on the following search results about 'Future of LLM in Tech Industry', pick the most 
relevant URL(s) and scrape them for deeper content.\n\nSearch Results:\nTitle: Future of Large Language Models - 
GeeksforGeeks\nURL: https://www.geeksforgeeks.org/machine-learning/future-of-large-language-models\nSnippet: # 
Future of Large Language Models. In the last few years, the development of artificial intelligence has been in 
significant demand, with the emergence of ****Large Language Models (LLMs)****. This streamlined model entails 
advanced machine learning methods, has transformed natural language procedur\n\n----\nTitle: Top LLM Trends 2025: 
What's the Future of LLMs - Turing\nURL: https://www.turing.com/resources/top-llm-trends\nSnippet: LLMs are 
evolving from basic assistants to [autonomous agents](https://www.turing.com/resources/ai-agentic-workflows) that 
can act independently. [Ethical AI](https://www.turing.com/resources/implementing-security-guardrails-for-llms) 
became a shared responsibility across developers, researchers, an\n\n----\nTitle: What comes after the LLMs? The AI
industry is organized around ...\nURL: 
https://www.facebook.com/technologyreview/posts/what-comes-after-the-llmsthe-ai-industry-is-organized-around-large-
language-mode/1283847436937737\nSnippet: ## MIT Technology Review's Post. ### **MIT Technology Review**. What comes
after the LLMs? The AI industry is organized around large language models, from tools to business models. But many 
researchers believe the next breakthroughs may not look like language models at all. Join us live tomorrow, 
Ma\n\n----\nTitle: Real-world Applications of Large Language Models | LLM Tutorial\nURL: 
https://www.youtube.com/watch?v=zQ-pLyp9Kp0\nSnippet: Real-world Applications of Large Language Models | LLM 
Tutorial | Future of Data and AI | Conference\nData Science Dojo\n122000 subscribers\n66 likes\n4744 views\n16 Mar 
2023\nCustom Large Language Models can improve your business in a number of ways. If you're not already using LLMs 
in your business, I e\n\n----\nTitle: AI LLM Evolution and the future of IT jobs! | by Seetaram N T - Medium\nURL: 
https://medium.com/@seetaram.nt/ai-llm-evolution-and-the-future-of-it-jobs-21440b36e25a\nSnippet: AI LLM Evolution 
and the future of IT jobs ... This can be used by people as reference to plan their career in IT Industry going 
forward!\n\n\nBased on the search results, it appears that the future of Large Language Models (LLMs) in the tech 
industry is expected to be shaped by several trends and developments. Some of the key points that can be inferred 
from the search results are:\n\n1. **Evolution of LLMs**: LLMs are expected to evolve from basic assistants to 
autonomous agents that can act independently.\n2. **Ethical AI**: The development of LLMs raises ethical concerns, 
and there is a growing need for developers, researchers, and organizations to prioritize ethical AI practices.\n3. 
**Real-world applications**: LLMs have the potential to improve various aspects of business and industry, including
customer service, content creation, and decision-making.\n4. **Career implications**: The evolution of LLMs may 
impact the job market, with some roles becoming redundant and new ones emerging.\n5. **Next breakthroughs**: 
Researchers believe that the next breakthroughs in AI may not come from language models but from other areas, such 
as computer vision or reinforcement learning.\n\nOverall, the future of LLMs in the tech industry is expected to be
shaped by a combination of technological advancements, ethical considerations, and real-world applications.",
        additional_kwargs={},
        response_metadata={},
        id='cf238e7d-c1c9-40c8-aceb-c84dedeb9843'
    ),
    AIMessage(
        content='',
        additional_kwargs={
            'tool_calls': [
                {
                    'id': 'fm2xa1akx',
                    'function': {
     

In [36]:
search_results_str

"Title: Future of Large Language Models - GeeksforGeeks\nURL: https://www.geeksforgeeks.org/machine-learning/future-of-large-language-models\nSnippet: # Future of Large Language Models. In the last few years, the development of artificial intelligence has been in significant demand, with the emergence of ****Large Language Models (LLMs)****. This streamlined model entails advanced machine learning methods, has transformed natural language procedur\n\n----\nTitle: Top LLM Trends 2025: What's the Future of LLMs - Turing\nURL: https://www.turing.com/resources/top-llm-trends\nSnippet: LLMs are evolving from basic assistants to [autonomous agents](https://www.turing.com/resources/ai-agentic-workflows) that can act independently. [Ethical AI](https://www.turing.com/resources/implementing-security-guardrails-for-llms) became a shared responsibility across developers, researchers, an\n\n----\nTitle: What comes after the LLMs? The AI industry is organized around ...\nURL: https://www.facebook.c

In [37]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [38]:
llm1 = ChatGroq(
    model="openai/gpt-oss-120b",     # 32k context
    temperature=0,
    max_tokens=2048,
)

In [39]:
#writer chain 

writer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert research writer. Write clear, structured and insightful reports."),
    ("human", """Write a detailed research report on the topic below.

Topic: {topic}

Research Gathered:
{research}

Structure the report as:
- Introduction
- Key Findings (minimum 3 well-explained points)
- Conclusion
- Sources (list all URLs found in the research)

Be detailed, factual and professional."""),
])

writer_chain = writer_prompt | llm1 | StrOutputParser()

In [40]:
def truncate_text(text: str, max_chars: int = 5500) -> str:
    """Truncate long text to avoid token limit errors"""
    if not text:
        return ""
    if len(text) > max_chars:
        return text[:max_chars] + "\n\n... [Content truncated due to length limit] ..."
    return text

In [41]:
# Before calling Reader Agent
state['search_results'] = truncate_text(str(state.get('search_results', '')), 5500)

# Before calling Writer / Critic
state['scraped_content'] = truncate_text(str(state.get('scraped_content', '')), 4000)

In [42]:
#step 3 - writer chain 

print("\n"+" ="*50)
print("step 3 - Writer is drafting the report ...")
print("="*50)

research_combined = (
    f"SEARCH RESULTS : \n {state['search_results']} \n\n"
    f"DETAILED SCRAPED CONTENT : \n {state['scraped_content']}"
)

state["report"] = writer_chain.invoke({
    "topic" : "Future of LLM in Tech Industry",
    "research" : research_combined
})

print("\n Final Report\n",state['report'])

= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =

step 3 - Writer is drafting the report ...

==================================================

Final Report
 **Future of Large Language Models (LLMs) in the Tech Industry**  
*Research Report*  

---

## 1. Introduction  

Large Language Models (LLMs) such as GPT‑4, Claude, LLaMA, and Gemini have moved from academic curiosities to core 
components of commercial products and services. Their ability to generate coherent text, reason over 
natural‑language prompts, and interface with external tools is reshaping software development, customer experience,
knowledge work, and even the structure of technology companies.  

This report synthesises publicly‑available material (see Sources) to outline the most consequential trends that 
will define the next 5‑10 years of LLM deployment in the tech sector, the challenges that must be addressed, and 
the strategic opportunities for organisations that adopt them early.

---

## 2. Key Findings  

### 2.1. From “Assistants” to Autonomous AI Agents  

| Aspect | Current State (2023‑24) | Expected Evolution (2025‑30) |
|--------|------------------------|------------------------------|
| **Interaction mode** | Prompt‑driven text generation; limited tool use. | Agents that can **plan**, **execute**, 
and **self‑correct** across APIs, databases, and UI elements without human‑in‑the‑loop prompting. |
| **Technical enablers** | Retrieval‑augmented generation (RAG), few‑shot prompting, tool‑calling APIs. | 
Integrated **reasoning loops**, **memory stores**, and **policy‑driven safety layers** that allow multi‑step 
problem solving (e.g., “book a flight, negotiate price, file expense report”). |
| **Business impact** | Customer‑service chatbots, code‑completion assistants. | End‑to‑end workflow automation 
(e.g., autonomous incident response, AI‑driven product design cycles). |

*Why it matters*: The shift to autonomous agents expands the value‑capture horizon from “speed‑up a human task” to 
“replace entire micro‑services with a single adaptable AI”. Companies that build **agentic platforms** (e.g., 
Turing’s AI‑agentic workflows) will become the backbone of next‑generation SaaS stacks.

---

### 2.2. Ethical AI, Governance, and “Guardrails” as Core Infrastructure  

1. **Regulatory pressure** – Emerging legislation in the EU (AI Act), US (Algorithmic Accountability Act), and 
China’s AI standards mandates transparency, risk assessments, and post‑deployment monitoring for LLM‑powered 
systems.  
2. **Security & privacy** – Prompt injection, model extraction, and data leakage are now recognized as 
high‑severity threats. Organizations are adopting **security guardrails** (prompt sanitisation, sandboxed 
inference, model watermarking).  
3. **Bias & fairness** – Systemic biases in training corpora continue to surface in downstream applications (e.g., 
hiring tools, legal drafting). The industry is moving toward **responsible fine‑tuning** pipelines that incorporate
human‑in‑the‑loop audits and synthetic data balancing.  

*Strategic implication*: Ethical AI is no longer an optional add‑on; it is a **critical component of product 
architecture**. Companies that embed governance tooling (audit logs, model‑level explainability, continuous bias 
monitoring) will gain faster market access and lower legal risk.

---

### 2.3. Real‑World, Domain‑Specific Deployments Accelerate  

| Domain | Representative Use‑Cases | Value Drivers |
|--------|--------------------------|---------------|
| **Enterprise Knowledge Management** | Retrieval‑augmented Q&A over internal docs, policy compliance checks. | 
Reduces time‑to‑knowledge, cuts support costs. |
| **Software Development** | Code generation, automated testing, bug triage, documentation synthesis. | Increases 
developer productivity, shortens release cycles. |
| **Customer Experience** | Hyper‑personalised chat, sentiment‑aware routing, multilingual support. | Improves NPS,
expands market reach. |
| **Healthcare & Life Sciences** | Clinical note summarisation, literature review assistants, drug‑discovery 
hypothesis generation. | Accelerates research, s

In [43]:
#critic_chain 

critic_prompt = ChatPromptTemplate.from_messages([
     ("system", "You are a sharp and constructive research critic. Be honest and specific."),
    ("human", """Review the research report below and evaluate it strictly.

Report:
{report}

Respond in this exact format:

Score: X/10

Strengths:
- ...
- ...

Areas to Improve:
- ...
- ...

One line verdict:
..."""),
])

critic_chain = critic_prompt | llm1 | StrOutputParser()

In [44]:

#critic report 

print("\n"+" ="*50)
print("step 4 - critic is reviewing the report ")
print("="*50)
state["feedback"] = critic_chain.invoke({
 "report":state['report']
})

print("\n critic report \n", state['feedback'])

= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =

step 4 - critic is reviewing the report

==================================================

critic report 
 **Score:** 7/10  

**Strengths:**  
- **Clear structure & navigation** – the report follows a logical flow (intro, findings, conclusion, sources) and 
uses tables and headings that make key points easy to scan.  
- **Comprehensive coverage of trends** – it identifies the most salient industry directions (agentic AI, 
governance, domain‑specific models, workforce impact, multimodal/hybrid AI) and ties each to concrete business 
implications.  
- **Action‑oriented insights** – each finding ends with a “why it matters” or “strategic implication” paragraph, 
giving readers a sense of what decisions to prioritize.  
- **Use of concrete examples** – references to specific platforms (LangChain, LlamaIndex, Turing) and use‑case 
categories help ground the discussion in real‑world practice.  
- **Citation of sources** – even though the sources are largely popular‑media pieces, the report lists them, 
showing an attempt at transparency.

**Areas to Improve:**  
- **Source credibility & depth** – the bibliography relies heavily on blog posts, a YouTube video, and a 
Facebook‑linked article; incorporating peer‑reviewed papers, industry white‑papers, or regulatory texts would 
strengthen authority.  
- **Quantitative backing** – the report makes many qualitative claims (e.g., “reduces support costs”, “increases 
developer productivity”) without data, market sizing, or case‑study metrics to substantiate impact.  
- **Technical specificity** – sections on “integrated reasoning loops”, “memory stores”, and “policy‑driven safety 
layers” remain high‑level; brief descriptions of concrete architectures (e.g., retrieval‑augmented generation 
pipelines, LoRA fine‑tuning, RLHF) would add rigor.  
- **Risk discussion is shallow** – while security and bias are mentioned, the report does not explore mitigation 
trade‑offs, cost implications, or failure‑mode scenarios that senior technologists expect.  
- **Formatting consistency** – mixed use of em‑dashes, bullet styles, and occasional typographical errors (e.g., 
“LLM‑as‑a‑service” vs “LLM‑as‑a‑service”) detract from professionalism.  
- **Future timeline vagueness** – the “2025‑30” horizon is broad; more granular milestones (e.g., “agentic 
orchestration in SaaS by 2026”) would help strategic planning.

**One line verdict:**  
A well‑organized overview of LLM trends, but it needs stronger, data‑driven evidence and more rigorous sources to 
be a truly actionable industry report.